# Test Case 7 — Scope Expansion in a Degraded-Data Environment

## Purpose

Demonstrates the **scope revision workflow** (scope expansion suggestion → analyst decision →
scope v1 run) and the `tskr_patterns` input pathway in a realistic degraded-data scenario.

## Scenario Summary

| Field | Value |
|---|---|
| **Event ID** | `E2026-04-20-001` |
| **System** | U1 RCP-C seal leakoff |
| **Primary observation** | Seal leakoff 2.4 gpm — above 2.0 gpm admin action level |
| **Data gap** | SOE and alarm log unavailable (process computer backup window) |
| **Scope expansion trigger** | HX-4C inlet temp: 4.5°F drift, 6h before leakoff — OUTSIDE initial scope |
| **Post-expansion primary** | `FM-SWHX4C-FOULING` (Category A, root) |

## Show-stopper

**Run 1:** Pipeline flags a temporal predecessor outside scope, flags missing SOE/alarm data
in the sensitivity table, and generates a scope expansion suggestion.

**Analyst approves** in one line of code.

**Run 2:** HX fouling candidate enters and becomes primary hypothesis. A prior event
(`EVT-U1-2024-0308`, same plant, same mechanism, confirmed root cause) is matched.
The before/after comparison is the show-stopper.

In [ ]:
from __future__ import annotations
import json, os, sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd().resolve()
FIXTURE_DIR   = NOTEBOOK_ROOT / "fixtures"
OUTPUT_DIR    = NOTEBOOK_ROOT / "rca_runs_case_007"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [
    os.path.abspath(os.path.join(os.getcwd(), "..", ".."))        ,
    os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")) ,
    os.path.abspath(os.path.join(os.getcwd(), "..", "shared"))    ,
]:
    if p not in sys.path:
        sys.path.insert(0, p)

from run_helpers import build_fixture_orchestrator, load_fixtures, run_rca, summarise_result
from assertion_helpers import (
    assert_candidate_present, assert_data_coverage_status,
    assert_scope_filter_status, assert_similar_event_match,
    run_assertion_table,
)
print("Imports OK. Fixture dir:", FIXTURE_DIR)

In [ ]:
fixtures = load_fixtures(FIXTURE_DIR)
print("Fixtures loaded:")
for k, v in fixtures.items():
    status = "present" if v is not None else "absent (deliberate gap)"
    print(f"  {k}: {status}")
print()
# Confirm SOE and alarm are absent (deliberate)
assert fixtures.get("soe_log") is None, "soe_log should be absent for TC-7"
assert fixtures.get("alarm_log") is None, "alarm_log should be absent for TC-7"
print("Data gap confirmed: soe_log and alarm_log absent (process computer backup window).")

## Run 1 — Discovery Mode (`active_scope_version=0`)

Initial scope: `U1-RCP-C` and its seal package only. The HX is outside scope.

**Expected:** scope filter not applied; expansion suggestion generated; SOE/alarm `not_assessed`.

In [ ]:
orc1 = build_fixture_orchestrator(OUTPUT_DIR / "run1", top_k_candidates=5, enable_ishikawa=False)
r1 = run_rca(orc1, fixtures)
print("=== Run 1 complete ===")
summarise_result(r1)

# Show scope management state
run_ctx = r1.get("run_context") or {}
scope_mgmt = (run_ctx.get("scope_management") or {})
suggestions = scope_mgmt.get("expansion_suggestions") or []
print(f"\nScope expansion suggestions: {len(suggestions)}")
for s in suggestions:
    print(f"  signal_id={s.get('signal_id')}  trigger={s.get('trigger_type')}  decision={s.get('decision')}")

# Show missing data coverage (path: run_manifest.artifacts.data_coverage_summary[source]["status"])
coverage = (r1.get("run_manifest") or {}).get("artifacts", {}).get("data_coverage_summary") or {}
for family in ["soe_log", "alarm_log"]:
    entry = coverage.get(family)
    status = entry.get("status") if isinstance(entry, dict) else entry
    print(f"  {family}: {status}")

In [ ]:
# Show sensitivity table from Run 1 — should flag missing SOE and alarm
sens = (r1.get("run_manifest") or {}).get("artifacts", {}).get("sensitivity_table") or {}
missing = sens.get("missing_sources_checked") or []
any_change = sens.get("any_ranking_change_possible")
print(f"Sensitivity table: missing_sources_checked = {missing}")
print(f"any_ranking_change_possible = {any_change}")

## Analyst Action — Accept Scope Expansion

The analyst reviews the scope expansion suggestion and accepts it:

```python
orc1.resolve_expansion_suggestion(
    signal_id="U1-SW-HX-4C",
    decision="accepted",
    analyst_rationale="HX inlet temperature precursor is plausible thermal path to seal degradation"
)
```

This advances `active_scope_version` to 1 and admits `U1-SWP-SEAL-WATER-HX-4C` failure modes
into the candidate set for Run 2.

In [ ]:
# Resolve expansion suggestion — the one-liner that makes the scope revision tangible
try:
    orc1.resolve_expansion_suggestion(
        signal_id="U1-SW-HX-4C",
        decision="accepted",
        analyst_rationale="HX inlet temperature precursor is plausible thermal path to seal degradation"
    )
    print("Scope expansion accepted. active_scope_version:",
          getattr(orc1, 'active_scope_version', 'see run_context in Run 2'))
except AttributeError:
    # Method signature may vary — update run_context directly if needed
    run_context = r1.get("run_context") or {}
    scope_mgmt = run_context.get("scope_management") or {}
    for s in scope_mgmt.get("expansion_suggestions", []):
        if "HX" in (s.get("signal_id") or ""):
            s["decision"] = "accepted"
            s["analyst_rationale"] = "HX inlet temperature precursor is plausible thermal path to seal degradation"
    print("Scope expansion accepted (via run_context update).")

## ⚠ Development Note — `resolve_expansion_suggestion` API

The cell above calls `orc1.resolve_expansion_suggestion(signal_id=..., decision=..., analyst_rationale=...)`.

**Before running this notebook live, verify the exact method signature** against the orchestrator implementation:

```python
# Check what's available on the orchestrator instance
[m for m in dir(orc1) if "scope" in m.lower() or "expansion" in m.lower() or "resolve" in m.lower()]
```

**Three things to confirm:**

1. **Method name** — the plan uses `resolve_expansion_suggestion`. If the actual method is named differently (e.g. `accept_scope_expansion`, `apply_scope_revision`), update cells 7 and 9 accordingly.

2. **Argument name** — the plan uses `signal_id` to identify the suggestion. The expansion suggestion in `run_context.scope_management.expansion_suggestions` may key on `component_id`, `signal_id`, or a suggestion index. Match the argument to whatever the method accepts.

3. **State transfer between Run 1 and Run 2** — the scope decision accepted on `orc1` must be visible to `orc2`. If the orchestrator carries scope state internally (via a shared `run_context` object), pass `run_context=r1["run_context"]` to `build_fixture_orchestrator` or `run_rca` for Run 2. If scope state is passed as a `run()` kwarg, update the `run_rca()` call in cell 9.

The `except AttributeError` fallback in cell 7 mutates the `r1["run_context"]` dict directly as a workaround if the method does not exist yet. If you use this path, pass the mutated context explicitly to Run 2:

```python
r2 = run_rca(orc2, fixtures, run_context_override=r1["run_context"])
# (adjust kwarg name to match the actual orchestrator.run() signature)
```

## Run 2 — Scope Active (`active_scope_version=1`)

HX-4C is now in scope. The fouling failure mode enters the candidate set.

**Expected:** scope filter applied; HX fouling ranked above seal wear; prior event matched.

In [ ]:
orc2 = build_fixture_orchestrator(OUTPUT_DIR / "run2", top_k_candidates=5, enable_ishikawa=True)
r2 = run_rca(orc2, fixtures)
print("=== Run 2 complete ===")
summarise_result(r2)

# Show scope filter status
scope_filter = (r2.get("run_manifest") or {}).get("artifacts", {}).get("scope_filter") or {}
print(f"\nscope_filter.applied = {scope_filter.get('applied')}")
print(f"scope_filter.approved_scope_version = {scope_filter.get('approved_scope_version')}")
print(f"scope_filter.filtered_count = {scope_filter.get('filtered_count')}")

# Show candidate ranking
candidates = (r2.get("causality_candidates") or {}).get("candidates") or []
print("\nCandidate ranking (Run 2):")
for c in sorted(candidates, key=lambda x: -(x.get("scores") or {}).get("composite", 0)):
    print(f"  [{c.get('failure_mode_id')}]  composite={c.get('scores', {}).get('composite', '?')}")

In [ ]:
# Show prior event match
similar = (r2.get("run_manifest") or {}).get("artifacts", {}).get("similar_event_list") or r2.get("similar_events") or {}
print(f"similar_event_list.any_plant_match = {similar.get('any_plant_match')}")
for evt in (similar.get("events") or [])[:3]:
    print(f"  {evt.get('event_id')} — {evt.get('description', '')[:80]}")

In [ ]:
assertions_r1 = [
    {"id": "A7-3", "desc": "Run 1: SOE not assessed",
     "fn": lambda r: assert_data_coverage_status(r, "soe_log", "not_assessed")},
    {"id": "A7-4", "desc": "Run 1: alarm not assessed",
     "fn": lambda r: assert_data_coverage_status(r, "alarm_log", "not_assessed")},
]
run_assertion_table(r1, assertions_r1, label="TC-7 Run 1 Assertions")

assertions_r2 = [
    {"id": "A7-7", "desc": "Run 2: HX fouling candidate present",
     "fn": lambda r: assert_candidate_present(r, "FM-SWHX4C-FOULING")},
    {"id": "A7-8", "desc": "Run 2: similar event plant match",
     "fn": lambda r: assert_similar_event_match(r, any_plant_match=True)},
]
run_assertion_table(r2, assertions_r2, label="TC-7 Run 2 Assertions")

In [ ]:
for label, result in [("run1", r1), ("run2", r2)]:
    out_path = OUTPUT_DIR / f"tc7_{label}_result.json"
    with open(out_path, "w", encoding="utf-8") as fh:
        json.dump(result, fh, indent=2, default=str)
    print(f"Saved: {out_path}")